# 03 — Feature Engineering & Selection

Phase 3, notebook 3 of 3. Per
[DECISIONS.md#feature-selection-vs-feature-extraction-clarified](../DECISIONS.md#feature-selection-vs-feature-extraction-clarified):
`feature_snapshots` holds the full *candidate* feature set; this notebook's
job is *selection* (mutual information / permutation importance / Boruta /
correlation pruning pick a subset) plus engineering ratio/interaction
features in pandas. Selection is recorded as a feature-list artifact, not
a schema migration.

In [1]:
import os
import sys
sys.path.insert(0, "../src")

import decimal

import numpy as np
import pandas as pd
import yaml
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OrdinalEncoder
from sqlalchemy import text

from churn.config import PROJECT_ROOT, get_app_config

# nbconvert executes with cwd = the notebook's own directory, but
# pydantic-settings resolves Settings' env_file=".env" relative to cwd --
# from notebooks/ that silently misses .env instead of erroring, and
# psycopg then fails with "no password supplied" instead of a clear
# config error. chdir to the project root before touching any settings.
os.chdir(PROJECT_ROOT)

from churn.db.engine import get_engine

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
RANDOM_SEED = None  # set below from cfg


def coerce_decimal_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """psycopg returns Postgres NUMERIC columns as Python Decimal objects
    (object dtype), not float -- coerce any object column that actually
    holds Decimals to float64 before any numeric/plotting/sklearn use."""
    for c in frame.columns:
        if frame[c].dtype == object and frame[c].map(lambda v: isinstance(v, decimal.Decimal)).any():
            frame[c] = pd.to_numeric(frame[c], errors="coerce")
    return frame


cfg = get_app_config()
RANDOM_SEED = cfg.synthetic.random_seed
engine = get_engine()

panel = pd.read_sql(
    text("SELECT * FROM feature_snapshots WHERE feature_set_version = :v"),
    engine,
    params={"v": cfg.feature_set.version},
)
panel["reference_date"] = pd.to_datetime(panel["reference_date"])
panel = coerce_decimal_columns(panel)
print("Full materialized panel:", panel.shape)

Full materialized panel: (87517, 39)


## 0. Drop the structurally-censored last cohort

Per
[DECISIONS.md#last-cohort-has-a-structural-zero-positive-rate](../DECISIONS.md#last-cohort-has-a-structural-zero-positive-rate)
and Notebook 02's confirmation: `reference_date == SNAPSHOT_DATE` can
never have a positive label (the synthetic churn-date generator can't
place a `churn_date` after `SNAPSHOT_DATE`), which is a censoring artifact,
not a real 0% hazard month. Every other cohort's positive rate is
consistent (1.3-2.0%), so only this single cohort is dropped from the
*modeling* panel used from here on — materialization (the full 87,517-row
table) is untouched.

In [2]:
snapshot_ts = pd.Timestamp(cfg.data.snapshot_date)
modeling_panel = panel[panel["reference_date"] < snapshot_ts].copy()
print(f"Dropped {len(panel) - len(modeling_panel)} rows (the {snapshot_ts.date()} cohort).")
print("Modeling panel:", modeling_panel.shape)
assert modeling_panel["label_churn_next_30d"].notna().all(), "Modeling panel has NULL labels -- expected every historical snapshot to have a matured, known outcome (see DECISIONS.md#label-lookahead-is-not-leakage)."
assert modeling_panel["label_churn_next_30d"].sum() > 0
assert (modeling_panel.groupby("reference_date")["label_churn_next_30d"].mean() > 0).all(), "A cohort in the modeling panel still has zero positives."
print("Confirmed: every remaining cohort has at least one positive, and every label is populated.")

Dropped 5174 rows (the 2024-06-01 cohort).
Modeling panel: (82343, 39)
Confirmed: every remaining cohort has at least one positive, and every label is populated.


## 1. Ratio & interaction features (business rationale)

Deliverable 4 asks for ratio and interaction features with an explicit
business rationale, computed here in pandas (not SQL) since they're cheap,
leakage-free derivations of already point-in-time-correct base columns —
see DECISIONS.md#feature-selection-vs-feature-extraction-clarified.

In [3]:
eps = 1e-6
df = modeling_panel.copy()

# 1. Spend intensity: average charge per month of relationship. Two
# customers with the same cumulative spend but very different tenure have
# very different risk profiles -- this normalizes for that.
df["charge_per_tenure_month"] = df["cumulative_charges_to_date"] / (df["tenure_months"] + 1)

# 2. Recent-vs-baseline ticket-velocity ratio: a spike detector. A
# customer whose last-30d ticket rate is far above their own 90d average
# is showing acute distress, not just chronically high support usage --
# this is the ratio form of the "ticket velocity" signal Deliverable 3
# asks feature engineering to extract. frequency_tickets_90d is a whole
# count that is legitimately 0 for most customers in most months, so the
# denominator needs a real smoothing prior (0.1 tickets/month), not a
# currency-scale eps -- 1e-6 there would turn "0 tickets in 90d, 1 ticket
# in 30d" into a nonsense ~1,000,000x outlier ratio instead of a large but
# sane one.
TICKET_RATE_PRIOR = 0.1
baseline_monthly_tickets = df["frequency_tickets_90d"] / 3.0
df["ticket_velocity_ratio_30_90"] = df["frequency_tickets_30d"] / (baseline_monthly_tickets + TICKET_RATE_PRIOR)

# 3. Payment-failure x contract-flexibility interaction: a month-to-month
# customer with a live payment-failure streak has both the financial
# distress signal AND no switching cost to leave -- the combination is
# plausibly worse than either alone.
df["is_month_to_month"] = (df["contract_type"] == "Month-to-month").astype(int)
df["failure_streak_x_month_to_month"] = df["payment_failure_streak"] * df["is_month_to_month"]

# 4. Normalized billing trend: raw billing_amount_slope_90d isn't
# comparable across customers on very different price tiers -- expressing
# it as a fraction of the customer's own monthly_charges makes a $2/month
# decline for a $20 plan and a $10/month decline for a $100 plan register
# as the same relative trend.
df["billing_slope_pct_of_charges"] = df["billing_amount_slope_90d"] / (df["monthly_charges"].astype(float) + eps)

# 5. High-value-at-risk flag: above-median spend AND a live payment-failure
# streak. Retention economics (config/config.yaml's business: block) cares
# more about losing a high-LTV customer than a low one, so this flags
# exactly the segment where a retention offer's economics are best.
median_charge = df["monthly_charges"].astype(float).median()
df["is_high_value_at_risk"] = (
    (df["monthly_charges"].astype(float) > median_charge) & (df["payment_failure_streak"] > 0)
).astype(int)

engineered_cols = [
    "charge_per_tenure_month", "ticket_velocity_ratio_30_90",
    "failure_streak_x_month_to_month", "billing_slope_pct_of_charges", "is_high_value_at_risk",
]
display(df[engineered_cols].describe())

,charge_per_tenure_month,ticket_velocity_ratio_30_90,failure_streak_x_month_to_month,billing_slope_pct_of_charges,is_high_value_at_risk
count,82343.000000,82343.000000,82343.000000,78416.000000,82343.000000
mean,60.456474,0.374346,0.031150,0.000001,0.026718
std,31.763366,0.807350,0.190209,0.000979,0.161258
min,0.000000,0.000000,0.000000,-0.009109,0.000000
25%,24.940768,0.000000,0.000000,-0.000389,0.000000
50%,64.441884,0.000000,0.000000,0.000001,0.000000
75%,87.666258,0.000000,0.000000,0.000392,0.000000
max,117.129412,2.830189,3.000000,0.009160,1.000000


## 2. Preprocessing for selection models

In [4]:
exclude = {"snapshot_id", "customer_id", "reference_date", "feature_set_version", "label_churn_next_30d", "created_at", "is_month_to_month"}
candidate_cols = [c for c in df.columns if c not in exclude]
X = df[candidate_cols].copy()
y = df["label_churn_next_30d"].astype(int)
groups = df["customer_id"]

# is_numeric_dtype(bool) is True in pandas (bool is numpy-numeric), so it
# must be excluded explicitly. Everything non-numeric -- object dtype,
# pandas' newer dedicated "str"/"string" dtype, categorical -- goes to
# cat_cols; checking `dtype == object` alone misses that newer string
# dtype entirely (this pandas defaults string columns to it, not object,
# confirmed by notebook 01's own df.info() showing "Dtype: str").
num_cols = [
    c for c in candidate_cols
    if pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])
]
cat_cols = [c for c in candidate_cols if c not in num_cols]

X_enc = X.copy()
encoders = {}
for c in cat_cols:
    enc = OrdinalEncoder()
    X_enc[c] = enc.fit_transform(X_enc[[c]].astype(str))
    encoders[c] = enc
for c in num_cols:
    X_enc[c] = X_enc[c].astype(float)
    X_enc[f"{c}_was_missing"] = X_enc[c].isna().astype(int)
    X_enc[c] = X_enc[c].fillna(X_enc[c].median())

missing_flag_cols = [c for c in X_enc.columns if c.endswith("_was_missing") and X_enc[c].sum() > 0]
model_cols = candidate_cols + missing_flag_cols
print(f"{len(candidate_cols)} candidate columns + {len(missing_flag_cols)} missingness flags = {len(model_cols)} model inputs")

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(X_enc, y, groups=groups))
print(f"Grouped train/val split: {len(train_idx)} train rows, {len(val_idx)} val rows, "
      f"{groups.iloc[train_idx].nunique()} / {groups.iloc[val_idx].nunique()} unique customers "
      f"(GroupKFold -- no customer appears in both, per the project's CV invariant).")

38 candidate columns + 6 missingness flags = 44 model inputs
Grouped train/val split: 65874 train rows, 16469 val rows, 5230 / 1308 unique customers (GroupKFold -- no customer appears in both, per the project's CV invariant).


## 3. Mutual information ranking

In [5]:
discrete_mask = [c in cat_cols or c in missing_flag_cols for c in model_cols]
mi = mutual_info_classif(X_enc[model_cols], y, discrete_features=discrete_mask, random_state=RANDOM_SEED)
mi_df = pd.DataFrame({"feature": model_cols, "mutual_info": mi}).sort_values("mutual_info", ascending=False).reset_index(drop=True)
display(mi_df)

,feature,mutual_info
0,frequency_charges_90d,0.022016
1,frequency_charges_30d,0.020221
2,frequency_charges_60d,0.020036
3,contract_type,0.009263
4,tenure_months,0.006593
5,tenure_bucket,0.005976
6,frequency_tickets_90d,0.005572
7,cumulative_charges_to_date,0.005322
8,recency_days_since_last_ticket,0.005109
9,frequency_tickets_30d,0.004961


## 4. Permutation importance (RandomForest baseline, grouped train/val split)

In [6]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
)
rf.fit(X_enc.iloc[train_idx][model_cols], y.iloc[train_idx])

from sklearn.metrics import average_precision_score
val_proba = rf.predict_proba(X_enc.iloc[val_idx][model_cols])[:, 1]
pr_auc = average_precision_score(y.iloc[val_idx], val_proba)
print(f"Baseline RF val PR-AUC: {pr_auc:.4f} (vs. {y.iloc[val_idx].mean():.4f} random-guess baseline)")

perm = permutation_importance(
    rf, X_enc.iloc[val_idx][model_cols], y.iloc[val_idx],
    n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1, scoring="average_precision",
)
perm_df = pd.DataFrame({
    "feature": model_cols, "perm_importance_mean": perm.importances_mean, "perm_importance_std": perm.importances_std,
}).sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)
display(perm_df)

Baseline RF val PR-AUC: 0.1184 (vs. 0.0169 random-guess baseline)


,feature,perm_importance_mean,perm_importance_std
0,contract_type,0.046623,0.009273
1,tenure_months,0.019331,0.006225
2,online_security,0.013168,0.004680
3,payment_method,0.011681,0.003247
4,online_backup,0.009930,0.006756
5,tenure_bucket,0.007670,0.003157
6,recency_days_since_last_ticket,0.007507,0.005251
7,cumulative_charges_to_date,0.006578,0.004081
8,tech_support,0.004974,0.003760
9,frequency_tickets_30d,0.004480,0.004007


## 5. Boruta-style shadow-feature selection (hand-rolled)

The `boruta` / `BorutaPy` package is unmaintained (last release ~2020) and
known to break against scikit-learn >= 1.3's API — a fragile dependency
for something whose core idea is simple enough to hand-roll and keep fully
testable, consistent with this project's existing preference (see
DECISIONS.md's `tech-stack` entry on hand-rolling drift statistics instead
of pulling in `evidently`). The algorithm: for each of several iterations,
shuffle every real feature to create a same-distribution "shadow" copy,
fit a random forest on real+shadow features together, and count how often
each real feature's importance beats the *best* shadow feature's
importance. A feature whose win-rate is unlikely under a fair coin flip
(binomial test) is "confirmed"; used consistently at chance is "rejected".

In [7]:
def boruta_shadow_selection(
    X_num: pd.DataFrame, y: pd.Series, n_iterations: int = 30, random_state: int = 42, alpha: float = 0.01,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    features = list(X_num.columns)
    wins = pd.Series(0, index=features)

    for i in range(n_iterations):
        shadow = X_num.apply(lambda col: rng.permutation(col.to_numpy()))
        shadow.columns = [f"shadow_{c}" for c in features]
        combined = pd.concat([X_num.reset_index(drop=True), shadow.reset_index(drop=True)], axis=1)

        rf_i = RandomForestClassifier(
            n_estimators=100, max_depth=6, min_samples_leaf=20,
            class_weight="balanced", random_state=int(rng.integers(0, 1_000_000)), n_jobs=-1,
        )
        rf_i.fit(combined, y.reset_index(drop=True))
        importances = pd.Series(rf_i.feature_importances_, index=combined.columns)
        max_shadow = importances[[f"shadow_{c}" for c in features]].max()
        beat_shadow = importances[features] > max_shadow
        wins += beat_shadow.astype(int)

    results = []
    for f in features:
        # Two-sided binomial test against p=0.5 (a real signal should win
        # more often than a coin flip, not just at chance).
        test = stats.binomtest(int(wins[f]), n_iterations, p=0.5, alternative="greater")
        verdict = "confirmed" if test.pvalue < alpha else ("rejected" if wins[f] <= n_iterations * 0.3 else "tentative")
        results.append({"feature": f, "wins": int(wins[f]), "n_iterations": n_iterations, "p_value": test.pvalue, "verdict": verdict})
    return pd.DataFrame(results).sort_values("wins", ascending=False).reset_index(drop=True)


# Numeric-only (Boruta's shuffle-and-compare logic is cleanest on
# already-numeric/encoded columns); run on the training fold only.
boruta_input = X_enc.iloc[train_idx][model_cols].astype(float)
boruta_df = boruta_shadow_selection(boruta_input, y.iloc[train_idx], n_iterations=30, random_state=RANDOM_SEED)
display(boruta_df)
print(boruta_df["verdict"].value_counts())

,feature,wins,n_iterations,p_value,verdict
0,contract_type,30,30,9.313226e-10,confirmed
1,online_backup,30,30,9.313226e-10,confirmed
2,online_security,30,30,9.313226e-10,confirmed
3,internet_service,30,30,9.313226e-10,confirmed
4,tech_support,30,30,9.313226e-10,confirmed
5,device_protection,30,30,9.313226e-10,confirmed
6,frequency_tickets_60d,30,30,9.313226e-10,confirmed
7,recency_days_since_last_ticket,30,30,9.313226e-10,confirmed
8,tenure_bucket,30,30,9.313226e-10,confirmed
9,tenure_months,30,30,9.313226e-10,confirmed


verdict
confirmed    27
rejected     14
tentative     3
Name: count, dtype: int64


## 6. Correlation pruning

In [8]:
corr = X_enc[num_cols].astype(float).corr(method="spearman")
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .rename("spearman_r")
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
)
high_corr = pairs[pairs["spearman_r"].abs() > 0.8].sort_values("spearman_r", key=abs, ascending=False)
display(high_corr)

mi_rank = mi_df.set_index("feature")["mutual_info"]
to_drop = set()
for _, row in high_corr.iterrows():
    a, b = row["feature_a"], row["feature_b"]
    loser = a if mi_rank.get(a, 0) < mi_rank.get(b, 0) else b
    to_drop.add(loser)
print(f"Correlation-pruned (lower-MI member of each |r|>0.8 pair): {sorted(to_drop)}")

,feature_a,feature_b,spearman_r
188,frequency_tickets_30d,ticket_velocity_ratio_30_90,0.994235
276,monetary_charges_60d,monetary_charges_90d,0.988700
307,monetary_charges_90d,charge_per_tenure_month,0.982446
283,monetary_charges_60d,charge_per_tenure_month,0.973679
251,monetary_charges_30d,monetary_charges_60d,0.969875
252,monetary_charges_30d,monetary_charges_90d,0.961483
358,billing_amount_slope_90d,billing_slope_pct_of_charges,0.961204
19,monthly_charges,charge_per_tenure_month,0.950654
259,monetary_charges_30d,charge_per_tenure_month,0.944620
11,monthly_charges,monetary_charges_60d,0.942433


Correlation-pruned (lower-MI member of each |r|>0.8 pair): ['billing_amount_slope_90d', 'charge_per_tenure_month', 'cumulative_charges_to_date', 'frequency_charges_60d', 'frequency_tickets_60d', 'monetary_charges_30d', 'monetary_charges_60d', 'monthly_charges', 'recency_days_since_last_ticket', 'ticket_velocity_ratio_30_90']


## 7. Final feature selection — combine methods

In [9]:
TOP_K = 15
top_mi = set(mi_df.head(TOP_K)["feature"])
top_perm = set(perm_df.head(TOP_K)["feature"])
boruta_ok = set(boruta_df[boruta_df["verdict"].isin(["confirmed", "tentative"])]["feature"])

vote_table = pd.DataFrame({"feature": model_cols})
vote_table["in_top_mi"] = vote_table["feature"].isin(top_mi)
vote_table["in_top_permutation"] = vote_table["feature"].isin(top_perm)
vote_table["boruta_confirmed_or_tentative"] = vote_table["feature"].isin(boruta_ok)
vote_table["correlation_pruned"] = vote_table["feature"].isin(to_drop)
vote_table["votes"] = vote_table[["in_top_mi", "in_top_permutation", "boruta_confirmed_or_tentative"]].sum(axis=1)

# Selection rule: at least 2 of 3 signal-detection methods agree, AND not
# the losing half of a highly-correlated pair.
vote_table["selected"] = (vote_table["votes"] >= 2) & (~vote_table["correlation_pruned"])
vote_table = vote_table.sort_values(["selected", "votes"], ascending=[False, False]).reset_index(drop=True)
display(vote_table)

selected_features = vote_table[vote_table["selected"]]["feature"].tolist()
print(f"\n{len(selected_features)} / {len(model_cols)} features selected:")
print(selected_features)

,feature,in_top_mi,in_top_permutation,boruta_confirmed_or_tentative,correlation_pruned,votes,selected
0,contract_type,True,True,True,False,3,True
1,online_security,True,True,True,False,3,True
2,tech_support,True,True,True,False,3,True
3,tenure_months,True,True,True,False,3,True
4,tenure_bucket,True,True,True,False,3,True
5,frequency_charges_90d,True,True,True,False,3,True
6,frequency_tickets_30d,True,True,True,False,3,True
7,frequency_tickets_90d,True,True,True,False,3,True
8,internet_service,False,True,True,False,2,True
9,online_backup,False,True,True,False,2,True



12 / 44 features selected:
['contract_type', 'online_security', 'tech_support', 'tenure_months', 'tenure_bucket', 'frequency_charges_90d', 'frequency_tickets_30d', 'frequency_tickets_90d', 'internet_service', 'online_backup', 'payment_method', 'monetary_charges_90d']


## 8. Write the feature-list artifact

Per
[DECISIONS.md#feature-selection-vs-feature-extraction-clarified](../DECISIONS.md#feature-selection-vs-feature-extraction-clarified),
selection is recorded as a feature-list artifact, not a schema change.
Written to `config/` rather than `artifacts/` because `artifacts/` is
git-ignored (trained model artifacts live in MLflow/MinIO, not git — see
`.gitignore`) and this needs to be a committed, reviewable, reproducible
input to Phase 4/5's `train.py`, exactly like `config/config.yaml`.

In [10]:
output_path = PROJECT_ROOT / "config" / "selected_features_v1.yaml"
artifact = {
    "feature_set_version": cfg.feature_set.version,
    "generated_by": "notebooks/03_feature_engineering.ipynb",
    "selection_rule": ">=2 of {top-15 mutual information, top-15 permutation importance, Boruta confirmed/tentative}, excluding the lower-MI half of any |Spearman r|>0.8 pair",
    "modeling_panel_note": "Excludes reference_date == SNAPSHOT_DATE cohort (structural zero-positive-rate censoring artifact).",
    "n_candidate_features": len(model_cols),
    "n_selected": len(selected_features),
    "selected_features": selected_features,
    "engineered_features_included": [c for c in engineered_cols if c in selected_features],
    "correlation_pruned": sorted(to_drop),
}
with open(output_path, "w", encoding="utf-8") as fh:
    yaml.safe_dump(artifact, fh, sort_keys=False)
print(f"Wrote {output_path}")
print(yaml.safe_dump(artifact, sort_keys=False))

Wrote C:\Users\11624\OneDrive - Indian School of Business\Documents\personal\ml_proj\config\selected_features_v1.yaml
feature_set_version: v1
generated_by: notebooks/03_feature_engineering.ipynb
selection_rule: '>=2 of {top-15 mutual information, top-15 permutation importance,
  Boruta confirmed/tentative}, excluding the lower-MI half of any |Spearman r|>0.8
  pair'
modeling_panel_note: Excludes reference_date == SNAPSHOT_DATE cohort (structural zero-positive-rate
  censoring artifact).
n_candidate_features: 44
n_selected: 12
selected_features:
- contract_type
- online_security
- tech_support
- tenure_months
- tenure_bucket
- frequency_charges_90d
- frequency_tickets_30d
- frequency_tickets_90d
- internet_service
- online_backup
- payment_method
- monetary_charges_90d
engineered_features_included: []
correlation_pruned:
- billing_amount_slope_90d
- charge_per_tenure_month
- cumulative_charges_to_date
- frequency_charges_60d
- frequency_tickets_60d
- monetary_charges_30d
- monetary_char

## Summary

- Engineered 5 ratio/interaction features with explicit business
  rationale (spend intensity, ticket-velocity spike ratio,
  failure-streak x contract-flexibility interaction, normalized billing
  trend, high-value-at-risk flag).
- Ran 4 independent selection methods: mutual information, permutation
  importance (grouped train/val split, PR-AUC scored), a hand-rolled
  Boruta shadow-feature procedure (the real `boruta` package is
  unmaintained and sklearn>=1.3-incompatible), and Spearman correlation
  pruning.
- Combined via a >=2-of-3 majority vote (Boruta/MI/permutation) minus
  correlation-pruned losers, written to
  `config/selected_features_v1.yaml` as the committed, reproducible input
  Phase 5's `train.py` will read — see DECISIONS.md for why this lives in
  `config/` and not `artifacts/`.
- Carries forward Notebook 02's recommendation: the modeling panel used
  throughout drops the structurally-censored `reference_date ==
  SNAPSHOT_DATE` cohort.